# OMNI-708 · The VIRAT archive on Colab's GPU, both streams

Runs the repo's own code, imported from a clone of the branch, never copied here: Algorithm 1
(`services/ingestion/app/pipeline.py`), SigLIP 2 (`services/ai_workers/app/encoder.py`),
InternVideo2 (`clip_encoder.py`), and the runner that ties them (`scripts/encode_virat_archive.py`).

1. **Setup:** clone the branch, install the pinned versions, mount Drive
2. **Load** the gate with the product's settings, and both encoders, pinned (D-30, D-33), float32
3. **Inputs:** the 275 VIRAT evaluation videos (`datasets/splits/virat.csv`, D-37) found on Drive
4. **The archive:** every frame gated, both vectors for every admitted frame, one file per video in
   `MyDrive/omnisight/embeddings/virat/` and a `manifest.json`. The product's 9 videos go first
5. **Check:** every video once, its file and counts agreeing
6. **The product's 9 videos:** their segments matched to archive vectors → `embeddings/product/`
7. **Agreement:** the laptop's frame vectors (OMNI-705) against the archive's
8. **Save** the summary to `MyDrive/omnisight/results/omni708_virat_archive.json`

**Run on a GPU:** Runtime → Change runtime type → GPU. Then Runtime → Run all.

**If the session is lost, or to test the resume:** Runtime → Run all again. Finished videos are skipped;
the one that was running is done again from its start.

In [ ]:
# 1 · Setup
BRANCH = "slice-07-search"
REPO = "https://github.com/OmniSight-team/omni-sight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

# The versions in uv.lock; Colab keeps its own GPU build of torch and torchvision.
!pip install -q "transformers==5.20.1" "huggingface-hub==2.2.0" "safetensors==0.8.0" "open-clip-torch==3.3.0" -e /content/omni-sight/services/common

# The repo's code straight from the clone (an editable install is only seen after a
# runtime restart, so the paths are added here too).
sys.path.insert(0, "/content/omni-sight/services/common")
sys.path.insert(0, "/content/omni-sight/services/ai_workers")
sys.path.insert(0, "/content/omni-sight/scripts")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 2 · Load the gate (Algorithm 1) and both encoders
from pathlib import Path
import encode_virat_archive as archive

REPO_DIR = Path("/content/omni-sight")
pipeline, GATE = archive.load_gate(REPO_DIR)
print("gate:", GATE)

from omnisight_common.settings import CommonSettings
from app.encoder import FrameTextEncoder, ensure_checkpoint
from app import clip_encoder
from app.clip_encoder import ClipTextEncoder

pins = {name: CommonSettings.model_fields[name].default for name in
        ("frame_model_id", "frame_model_revision", "clip_model_id", "clip_model_revision")}
print("pins:", pins)

DRIVE = Path("/content/drive/MyDrive/omnisight")
MODELS = DRIVE / "models" / "pretrained"   # cached on Drive at the pinned revision (OMNI-706)
VIRAT = DRIVE / "data" / "virat"
ARCHIVE = DRIVE / "embeddings" / "virat"
PRODUCT = DRIVE / "embeddings" / "product"

siglip_dir = ensure_checkpoint(pins["frame_model_id"], pins["frame_model_revision"], MODELS / "siglip2-so400m-patch14-384")
siglip = FrameTextEncoder(siglip_dir, device="cuda")
iv2_dir = ensure_checkpoint(pins["clip_model_id"], pins["clip_model_revision"], MODELS / "internvideo2-clip-s", clip_encoder.ALLOW_PATTERNS)
iv2 = ClipTextEncoder(iv2_dir, device="cuda")
print(f"both encoders on {torch.cuda.get_device_name(0)}, {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

In [ ]:
# 3 · The 275 VIRAT evaluation videos (D-37), found on Drive; the product's 9 first
KEYS = archive.evaluation_keys(REPO_DIR, first=archive.product_keys(REPO_DIR))
missing = [k for k in KEYS if not (VIRAT / "videos_original" / f"{k}.mp4").is_file()]
print(f"{len(KEYS)} evaluation videos, {len(KEYS) - len(missing)} on Drive, {len(missing)} missing")
assert len(KEYS) == 275 and not missing, missing[:10]

In [ ]:
# 4 · The archive: gate every frame, encode both streams of every admitted frame, one file per video
import time
SESSION = time.strftime("%Y-%m-%dT%H:%M:%S") + " " + torch.cuda.get_device_name(0)
manifest = archive.run(
    KEYS, VIRAT / "videos_original", ARCHIVE, pipeline, GATE, pins,
    siglip, clip_encoder, iv2, Path("/content/scratch"), session=SESSION,
)

In [ ]:
# 5 · Check: every video once, with its file, the counts agreeing; nothing missing or failed
report = archive.check(KEYS, ARCHIVE)
print({k: v for k, v in report.items() if k not in ("missing",)}, "missing:", len(report["missing"]))
assert not report["missing"] and not report["problems"] and not report["failed"] and not report["files_not_in_manifest"]
hours = report["seconds"] / 3600
print(f"{report['videos']} videos, {report['frames_decoded']:,} frames decoded, {report['admitted']:,} admitted "
      f"({report['motion']:,} motion, {report['heartbeat']:,} heartbeat), {hours:.2f} h of runs, "
      f"{report['frames_decoded'] / report['seconds']:.0f} frames/s decoded and gated, "
      f"{report['admitted'] / report['seconds']:.2f} admitted frames/s encoded (both streams)")

In [ ]:
# 6 · The product's 9 VIRAT videos: their segments (scripts/export_segments_for_colab.py, run on
# the laptop) matched to the archive by clip_key and frame_index; every one must be found
import json
match = archive.match_product(PRODUCT / "segments.json", ARCHIVE, PRODUCT)
for video_id, v in match["videos"].items():
    print(video_id, v)
print("unmatched:", len(match["unmatched"]), match["unmatched"][:5])
assert not match["unmatched"]

In [ ]:
# 7 · Agreement: the laptop's SigLIP 2 vectors (OMNI-705, CPU, the product's JPEG) against the
# archive's (GPU, the JPEG written here from the same frame); at least 0.999 each, or explained
import numpy as np
cos = np.array([a["cosine"] for a in match["agreement"]])
print(f"{len(cos)} frames: min {cos.min():.6f}, median {np.median(cos):.6f}, max {cos.max():.6f}; "
      f"{int((cos >= 0.999).sum())} of {len(cos)} at least 0.999")

In [ ]:
# 8 · Save the summary to Drive
out = DRIVE / "results" / "omni708_virat_archive.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({
    "commit": COMMIT, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "pins": pins, "gate": GATE, "check": report,
    "match": {"videos": match["videos"], "unmatched": match["unmatched"]},
    "agreement": match["agreement"],
}, indent=1, default=str))
print(out)